# Movie Recommendation System

Matrix-factorization recommender (SVD) on real MovieTweetings ratings (0-10 scale): predicts how a user would rate unseen movies and serves a top-5 list.


In [1]:

import io, urllib.request
import pandas as pd, numpy as np
# MovieTweetings: real movie ratings shared publicly on Twitter (0-10 scale).
# We keep the active-user / popular-movie subset so the matrix fits in memory.
def get(u):
    req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req).read()
r = pd.read_csv(io.StringIO(get("https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat").decode("utf-8", "replace")),
                sep="::", engine="python", header=None, names=["user", "item", "rating", "ts"])
tu, ti = 50, 50
for _ in range(5):
    uc = r["user"].value_counts(); ic = r["item"].value_counts()
    keep_u, keep_i = uc[uc >= tu].index, ic[ic >= ti].index
    if len(keep_u) <= 6000 and len(keep_i) <= 4000: break
    tu = int(tu * 1.5); ti = int(ti * 1.5)
ratings = r[r["user"].isin(keep_u) & r["item"].isin(keep_i)].copy()
try:
    mv = pd.read_csv(io.StringIO(get("https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat").decode("utf-8", "replace")),
                     sep="::", engine="python", header=None, names=["item", "title", "genres"])
    titles = dict(zip(mv["item"], mv["title"]))
except Exception:
    titles = {}
print("ratings:", len(ratings), "| users:", ratings["user"].nunique(), "| movies:", ratings["item"].nunique())


ratings: 413402 | users: 3976 | movies: 2982


In [2]:

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
r_train, r_test = train_test_split(ratings, test_size=0.2, random_state=42)
users = sorted(ratings["user"].unique()); items = sorted(ratings["item"].unique())
uidx = {u: k for k, u in enumerate(users)}; iidx = {i: k for k, i in enumerate(items)}
R = np.zeros((len(users), len(items)))
for row in r_train.itertuples(): R[uidx[row.user], iidx[row.item]] = row.rating
means = np.array([R[k][R[k] > 0].mean() if (R[k] > 0).any() else ratings["rating"].mean() for k in range(len(users))])
Rc = np.where(R > 0, R - means[:, None], 0)
svd = TruncatedSVD(n_components=30, random_state=42)
pred_all = (svd.fit_transform(Rc) @ svd.components_) + means[:, None]
preds = [float(np.clip(pred_all[uidx[r.user], iidx[r.item]], 0, 10)) for r in r_test.itertuples()]
rmse = float(np.sqrt(mean_squared_error(list(r_test["rating"]), preds)))
baseline = float(np.sqrt(mean_squared_error(list(r_test["rating"]), [ratings["rating"].mean()] * len(r_test))))
print("Test RMSE:", round(rmse, 4), "| mean-rating baseline RMSE:", round(baseline, 4), "(ratings are 0-10)")


Test RMSE: 1.5138 | mean-rating baseline RMSE: 1.7307 (ratings are 0-10)


In [3]:

example_user = users[0]
seen = set(r_train[r_train["user"] == example_user]["item"])
cands = [(items[k], float(pred_all[uidx[example_user], k])) for k in range(len(items)) if items[k] not in seen]
print(f"Top 5 recommendations for user {example_user}:")
for item_id, p in sorted(cands, key=lambda t: -t[1])[:5]:
    print(f"  {titles.get(item_id, item_id)}  (predicted {p:.2f} / 10)")


Top 5 recommendations for user 39:
  X-Men: Days of Future Past (2014)  (predicted 8.55 / 10)
  Avengers: Endgame (2019)  (predicted 8.54 / 10)
  The Imitation Game (2014)  (predicted 8.50 / 10)
  Fury (2014)  (predicted 8.48 / 10)
  1917 (2019)  (predicted 8.47 / 10)


## Result

See README.md for the verified RMSE. Matrix factorization (SVD) learns hidden taste factors from ratings alone — the same core idea behind classic recommender systems.
